                                                # 02 - Préparation du dataset pour Power BI

                                                    **Projet DWFA, Drinking Water For All**

## Objectif

Ce notebook construit, à partir des cinq fichiers nettoyés dans `01_exploration.ipynb`, six fichiers CSV prêts à être importés dans Power BI. Les jointures entre les tables sont réalisées dans l'outil de restitution et non dans ce notebook, conformément à la consigne de préparation du modèle de données.

Les six fichiers exportés sont les suivants :
1. `water_services_Final.csv`
2. `population_Final.csv`
3. `political_stability_Final.csv`
4. `mortality_Final.csv`
5. `region_country_Final.csv`
6. `dim_country_year.csv`, la table pivot technique construite dans ce notebook

## Pourquoi aucun fichier fusionné n'est produit ?

Trois des cinq tables sources comportent plusieurs lignes par pays et par année. Water Services distingue Rural, Total et Urbain, Population distingue Total, Male, Female, Rural et Urban, et Mortality distingue Female, Male et Total. Une fusion à plat sur `Country` et `Year` produirait donc un produit cartésien partiel : chaque ligne Rural de Water Services se combinerait avec chaque ligne Homme, Femme et Total de Population et de Mortality pour un même pays et une même année, ce qui créerait des doublons silencieux. Dans un outil destiné à éclairer une décision de financement, ce type de biais de comptage est particulièrement problématique, car il fausse tout indicateur agrégé sans que l'erreur ne soit visible à l'écran.

## Modèle retenu :

Une table pivot nommée `dim_country_year`, munie d'une clé unique, se trouve au centre du modèle. Les quatre tables de données s'y rattachent selon une relation plusieurs vers un, et la table `region_country` s'y raccroche via la seule colonne `Country`, cette dernière table ne comportant pas de dimension temporelle.

## Imports

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

---
## 1. Services d'eau, `water_services_clean.csv`  
---

Les trois granularités Rural, Urbain et Total sont conservées sans filtrage, décision déjà justifiée lors de l'exploration : le Domaine 1, création de services, nécessite de croiser le taux d'accès à l'eau avec le taux de population urbaine, et la distinction rural/urbain est une information directement exploitable pour cette analyse plutôt qu'un bruit à écarter.

Cette conservation a une conséquence directe sur la structure de la table : puisque trois lignes partagent le même pays et la même année, la colonne `Key` construite plus bas ne sera pas unique dans ce fichier. C'est précisément cette non-unicité qui justifie la construction d'une table pivot séparée, `dim_country_year`, plutôt qu'une relation directe entre les quatre tables de données.

In [2]:
df_water_services = pd.read_csv("../DATA/water_services_clean.csv")

print("Shape :", df_water_services.shape)
print("Granularités :", df_water_services["Granularity"].unique())
print("Valeurs manquantes :")
print(df_water_services.isnull().sum())

df_water_services.head()

Shape : (10476, 5)
Granularités : <StringArray>
['Rural', 'Total', 'Urban']
Length: 3, dtype: str
Valeurs manquantes :
Year                                                              0
Country                                                           0
Granularity                                                       0
Population using at least basic drinking-water services (%)    1061
Population using safely managed drinking-water services (%)    7190
dtype: int64


,Year,Country,Granularity,Population using at least basic drinking-water services (%),Population using safely managed drinking-water services (%)
0,2000,Afghanistan,Rural,21.61913,NaN
1,2000,Afghanistan,Total,27.77190,NaN
2,2000,Afghanistan,Urban,49.48745,NaN
3,2000,Albania,Rural,81.78472,NaN
4,2000,Albania,Total,87.86662,49.29324


### Renommage des colonnes

Les noms de colonnes issus de la source d'origine sont longs et peu adaptés à un usage dans Power BI. Ils sont raccourcis en `pct_basic_water` et `pct_safely_managed_water`. Ce renommage ne modifie aucune valeur, il ne porte que sur les en-têtes de colonnes.

In [3]:
col_basic = "Population using at least basic drinking-water services (%)"
col_safe = "Population using safely managed drinking-water services (%)"

df_water_services = df_water_services.rename(
    columns={col_basic: "pct_basic_water", col_safe: "pct_safely_managed_water"},
    errors="raise",
)

### Construction de la clé de jointure

Une colonne `Key` est ajoutée, obtenue par concaténation de `Country` et de `Year`. Cette clé servira à relier la table `water_services` à la table pivot `dim_country_year` une fois les deux tables importées dans Power BI.

In [4]:
df_water_services["Key"] = df_water_services["Country"] + "_" + df_water_services["Year"].astype(str)
df_water_services = df_water_services[
    ["Key", "Country", "Year", "Granularity", "pct_basic_water", "pct_safely_managed_water"]
]

df_water_services.head()

,Key,Country,Year,Granularity,pct_basic_water,pct_safely_managed_water
0,Afghanistan_2000,Afghanistan,2000,Rural,21.61913,NaN
1,Afghanistan_2000,Afghanistan,2000,Total,27.77190,NaN
2,Afghanistan_2000,Afghanistan,2000,Urban,49.48745,NaN
3,Albania_2000,Albania,2000,Rural,81.78472,NaN
4,Albania_2000,Albania,2000,Total,87.86662,49.29324


---
## 2. Population, `population_clean.csv`  
---

Les cinq granularités Total, Male, Female, Rural et Urban sont conservées, pour la même raison que la table Water Services : cette information sert directement le calcul du taux de population urbaine, utilisé comme indicateur du Domaine 1.

In [5]:
df_population = pd.read_csv("../DATA/population_clean.csv")

print("Shape :", df_population.shape)
print("Granularités :", df_population["Granularity"].unique())
print("Valeurs manquantes :", df_population["Population"].isnull().sum())

df_population["Key"] = df_population["Country"] + "_" + df_population["Year"].astype(str)
df_population = df_population[["Key", "Country", "Year", "Granularity", "Population"]]

df_population.head()

Shape : (20914, 5)
Granularités : <StringArray>
['Total', 'Male', 'Female', 'Rural', 'Urban']
Length: 5, dtype: str
Valeurs manquantes : 0


,Key,Country,Year,Granularity,Population
0,Afghanistan_2000,Afghanistan,2000,Total,20779.953
1,Afghanistan_2000,Afghanistan,2000,Male,10689.508
2,Afghanistan_2000,Afghanistan,2000,Female,10090.449
3,Afghanistan_2000,Afghanistan,2000,Rural,15657.474
4,Afghanistan_2000,Afghanistan,2000,Urban,4436.282


---
## 3. Stabilité politique, `political_stability_clean.csv`
---
Cette table ne comporte qu'une seule granularité, Total, un score de stabilité politique ne se déclinant ni par genre ni par zone géographique. La colonne `Key` y est donc naturellement unique, sans qu'aucun traitement particulier ne soit nécessaire pour l'assurer.

In [6]:
df_political_stability = pd.read_csv("../DATA/political_stability_clean.csv")

print("Shape :", df_political_stability.shape)
print("Granularités :", df_political_stability["Granularity"].unique())
print("Valeurs manquantes :", df_political_stability["Political_Stability"].isnull().sum())

df_political_stability["Key"] = (
    df_political_stability["Country"] + "_" + df_political_stability["Year"].astype(str)
)
df_political_stability = df_political_stability[
    ["Key", "Country", "Year", "Granularity", "Political_Stability"]
]

df_political_stability.head()

Shape : (3526, 4)
Granularités : <StringArray>
['Total']
Length: 1, dtype: str
Valeurs manquantes : 0


,Key,Country,Year,Granularity,Political_Stability
0,Afghanistan_2000,Afghanistan,2000,Total,-2.44
1,Afghanistan_2002,Afghanistan,2002,Total,-2.04
2,Afghanistan_2003,Afghanistan,2003,Total,-2.20
3,Afghanistan_2004,Afghanistan,2004,Total,-2.30
4,Afghanistan_2005,Afghanistan,2005,Total,-2.07


---
## 4. Mortalité liée à l'eau, `mortality_clean.csv`  
---

Cette table ne couvre que l'année 2016, une limite déjà identifiée lors de l'exploration et à rappeler explicitement en soutenance : elle sera présentée comme une preuve de rigueur analytique plutôt que dissimulée. Les granularités Female, Male et Total sont conservées, selon la même logique de préservation de l'information que pour Population et Water Services : la répartition par genre est un axe d'analyse à part entière, pas une donnée à agréger prématurément.

Un contrôle de cohérence des noms de colonnes est ajouté avant le renommage, afin d'obtenir un message d'erreur explicite en cas d'écart entre le nom attendu et le nom réellement présent dans le fichier source, plutôt qu'une erreur silencieuse qui ne se manifesterait que plus loin dans le notebook.

In [7]:
df_mortality = pd.read_csv("../DATA/mortality_clean.csv")

col_rate = "Mortality rate attributed to exposure to unsafe WASH services"
col_deaths = "WASH deaths"

print("Shape :", df_mortality.shape)
print("Colonnes disponibles :", df_mortality.columns.tolist())
print("Années :", df_mortality["Year"].unique())
print("Valeurs manquantes :")
print(df_mortality.isnull().sum())

assert col_rate in df_mortality.columns, (
    f"Colonne absente : {col_rate!r}. "
    f"Colonnes présentes : {df_mortality.columns.tolist()}"
)
assert col_deaths in df_mortality.columns, (
    f"Colonne absente : {col_deaths!r}. "
    f"Colonnes présentes : {df_mortality.columns.tolist()}"
)

Shape : (549, 5)
Colonnes disponibles : ['Year', 'Country', 'Granularity', 'Mortality rate attributed to exposure to unsafe WASH services', 'WASH deaths']
Années : [2016]
Valeurs manquantes :
Year                                                               0
Country                                                            0
Granularity                                                        0
Mortality rate attributed to exposure to unsafe WASH services      0
WASH deaths                                                      366
dtype: int64


Le renommage est ensuite appliqué avec l'option `errors="raise"`, qui interrompt l'exécution avec un message explicite si l'une des deux colonnes attendues n'est pas trouvée, plutôt que de l'ignorer silencieusement comme le fait le comportement par défaut de `rename`.

In [8]:
df_mortality = df_mortality.rename(
    columns={col_rate: "mortality_rate_per_100k", col_deaths: "wash_deaths"},
    errors="raise",
)
df_mortality["Key"] = df_mortality["Country"] + "_" + df_mortality["Year"].astype(str)
df_mortality = df_mortality[
    ["Key", "Country", "Year", "Granularity", "mortality_rate_per_100k", "wash_deaths"]
]

df_mortality.head()

,Key,Country,Year,Granularity,mortality_rate_per_100k,wash_deaths
0,Afghanistan_2016,Afghanistan,2016,Female,15.31193,NaN
1,Afghanistan_2016,Afghanistan,2016,Male,12.61297,NaN
2,Afghanistan_2016,Afghanistan,2016,Total,13.92067,4824.353
3,Albania_2016,Albania,2016,Female,0.12552,NaN
4,Albania_2016,Albania,2016,Male,0.20650,NaN


---
## 5. Correspondance pays et région, `region_country_clean.csv`  
---

Cette table constitue une dimension fixe, sans année associée. Elle se joint uniquement sur `Country`, une colonne `Key` n'aurait donc pas de sens dans ce fichier.

In [9]:
df_region_country = pd.read_csv("../DATA/region_country_clean.csv")
df_region_country = df_region_country.rename(
    columns={"COUNTRY (DISPLAY)": "Country", "REGION (DISPLAY)": "Region"},
    errors="raise",
)

print("Shape :", df_region_country.shape)
print("Régions :", df_region_country["Region"].unique())

df_region_country.head()

Shape : (194, 2)
Régions : <StringArray>
[               'Europe',       'Western Pacific', 'Eastern Mediterranean',
       'South-East Asia',                'Africa',              'Americas']
Length: 6, dtype: str


,Region,Country
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia
3,Western Pacific,Australia
4,Europe,Austria


---
## 6. Table pivot technique, `dim_country_year`  
---

Water Services, Population et Mortality ont plusieurs lignes par pays et par année (une par granularité : Rural/Urbain/Total ou Female/Male/Total). Leur colonne `Key` risque donc de ne pas être unique. Or Power BI impose qu'au moins l'un des deux côtés d'une relation ait une clé unique. L'idée : construire une table séparée, avec une seule ligne par pays et par année, qui servira de pivot central entre les quatre tables de données.

In [10]:
print("Key unique dans water_services :", df_water_services["Key"].is_unique)
print("Key unique dans population :", df_population["Key"].is_unique)
print("Key unique dans mortality :", df_mortality["Key"].is_unique)
print("Key unique dans political_stability :", df_political_stability["Key"].is_unique)

Key unique dans water_services : False
Key unique dans population : False
Key unique dans mortality : False
Key unique dans political_stability : True


La vérification confirme l'hypothèse : `Key` n'est pas unique dans Water Services, Population et Mortality, seule Political Stability l'est. La table pivot est construite à partir de `population`, qui couvre la période la plus large (2000 à 2018) et le plus grand nombre de pays parmi les cinq sources. Le contrôle qui suit montrera que ce choix, pourtant intuitif, ne suffit pas.

In [11]:
df_dim_country_year = (
    df_population[["Country", "Year"]].drop_duplicates().reset_index(drop=True)
)
df_dim_country_year["Key"] = (
    df_dim_country_year["Country"] + "_" + df_dim_country_year["Year"].astype(str)
)
df_dim_country_year = df_dim_country_year[["Key", "Country", "Year"]]

assert df_dim_country_year["Key"].is_unique, "La clé n'est pas unique dans dim_country_year"
print("dim_country_year créée,", df_dim_country_year.shape[0], "combinaisons pays-année uniques")

df_dim_country_year.head()

dim_country_year créée, 4430 combinaisons pays-année uniques


,Key,Country,Year
0,Afghanistan_2000,Afghanistan,2000
1,Afghanistan_2001,Afghanistan,2001
2,Afghanistan_2002,Afghanistan,2002
3,Afghanistan_2003,Afghanistan,2003
4,Afghanistan_2004,Afghanistan,2004


>### **Contrôle avant export : toutes les lignes ont-elles un pays et une année reconnus par la table pivot ?**

Avant l'export, il convient de vérifier que les clés des tables Water Services, Mortality et Political Stability existent bien dans `dim_country_year`. Une clé absente du pivot créerait une ligne orpheline une fois importée dans Power BI, invisible dans les visuels filtrés par pays et par année.

In [12]:
keys_dim = set(df_dim_country_year["Key"])

for name, df in [
    ("Water Services", df_water_services),
    ("Population", df_population),
    ("Political Stability", df_political_stability),
    ("Mortality", df_mortality),
]:
    orphelines = set(df["Key"]) - keys_dim
    print(f"{name}, clés orphelines : {len(orphelines)}")
    if orphelines:
        print("  Exemples :", list(orphelines)[:5])


Water Services, clés orphelines : 36
  Exemples : ['Republic of North Macedonia_2016', 'South Sudan_2003', 'Republic of North Macedonia_2006', 'South Sudan_2000', 'Republic of North Macedonia_2013']
Population, clés orphelines : 0
Political Stability, clés orphelines : 1
  Exemples : ['South Sudan_2011']
Mortality, clés orphelines : 1
  Exemples : ['Republic of North Macedonia_2016']


### **Conclusion :**

36 clés orphelines sur Water Services, 1 sur Mortality, 1 sur Political Stability. En regardant précisément quels pays sont concernés, deux causes distinctes :

- **Macédoine du Nord** : `Water Services` et `Mortality` désignent le pays `Republic of North Macedonia`, alors que `Population` et `Political Stability` utilisent `North Macedonia`. Erreur de nom, corrigeable. Elle explique 18 des clés orphelines de Water Services et l'unique clé de Mortality.
- **Monténégro et Soudan du Sud** : cas différent. `Population` ne compte ces deux pays qu'à partir de leur indépendance, 2006 et 2012, alors que l'OMS reconstitue les données d'accès à l'eau sur le territoire actuel depuis 2000. Il n'y a pas de nom à corriger, les deux sources sont justes : elles ne couvrent simplement pas la même période. Le Soudan du Sud 2011 relève de la même cause, `Political Stability` disposant d'une valeur dès l'année de l'indépendance.

**Décision : l'erreur de nom est corrigée. La différence de périmètre est traitée en élargissant la table pivot, afin qu'aucune ligne de donnée ne devienne inaccessible dans Power BI.**


In [13]:
# Erreur de nom : Macédoine du Nord
df_water_services["Country"] = df_water_services["Country"].replace(
    "Republic of North Macedonia", "North Macedonia"
)
df_mortality["Country"] = df_mortality["Country"].replace(
    "Republic of North Macedonia", "North Macedonia"
)

# Country a changé sur ces deux tables : Key doit être reconstruite,
# elle ne se met pas à jour automatiquement.
df_water_services["Key"] = df_water_services["Country"] + "_" + df_water_services["Year"].astype(str)
df_mortality["Key"] = df_mortality["Country"] + "_" + df_mortality["Year"].astype(str)


La table pivot est reconstruite à partir des noms corrigés. Sa construction change également de périmètre : jusqu'ici elle était extraite de la seule table `population`, ce qui laissait sans correspondance les couples pays-année couverts par les autres tables mais absents de celle-ci. Une dimension de référence doit couvrir l'intégralité des couples présents dans les tables qu'elle relie, faute de quoi les lignes non couvertes deviennent inaccessibles dans l'outil de restitution. Elle est donc construite sur l'union des quatre tables de données.

In [14]:
# Une dimension de référence doit couvrir tous les couples pays-année des tables qu'elle relie.
paires = pd.concat([
    df_water_services[["Country", "Year"]],
    df_population[["Country", "Year"]],
    df_political_stability[["Country", "Year"]],
    df_mortality[["Country", "Year"]],
])

df_dim_country_year = (
    paires.drop_duplicates().sort_values(["Country", "Year"]).reset_index(drop=True)
)
df_dim_country_year["Key"] = (
    df_dim_country_year["Country"] + "_" + df_dim_country_year["Year"].astype(str)
)
df_dim_country_year = df_dim_country_year[["Key", "Country", "Year"]]

assert df_dim_country_year["Key"].is_unique, "La clé n'est pas unique dans dim_country_year"
print("dim_country_year reconstruite,", df_dim_country_year.shape[0], "combinaisons pays-année uniques")
print("Pays couverts :", df_dim_country_year["Country"].nunique())


dim_country_year reconstruite, 4448 combinaisons pays-année uniques
Pays couverts : 237


In [15]:
keys_dim = set(df_dim_country_year["Key"])

for name, df in [
    ("Water Services", df_water_services),
    ("Population", df_population),
    ("Political Stability", df_political_stability),
    ("Mortality", df_mortality),
]:
    orphelines = set(df["Key"]) - keys_dim
    print(f"{name}, clés orphelines : {len(orphelines)}")
    if orphelines:
        print("  Exemples :", list(orphelines)[:5])


Water Services, clés orphelines : 0
Population, clés orphelines : 0
Political Stability, clés orphelines : 0
Mortality, clés orphelines : 0


### Conclusion :

Les clés orphelines tombent à zéro sur les quatre tables de données. La correction du nom de la Macédoine du Nord a réglé dix-neuf clés, dix-huit sur Water Services et une sur Mortality ; l'élargissement du périmètre de la table pivot a réglé les dix-neuf restantes, le Monténégro avant 2006 et le Soudan du Sud avant 2012.

Ces couples ne sont pas une anomalie. `population` compte ces deux pays à partir de leur indépendance, alors que les données d'accès à l'eau sont reconstituées par l'OMS sur le territoire actuel depuis 2000. Les deux sources sont justes, elles répondent à deux questions différentes. La table pivot retient la couverture la plus large des deux, de sorte qu'aucune ligne de donnée ne soit perdue : sur ces années, les indicateurs d'accès à l'eau sont disponibles et la population ne l'est pas, ce que les visuels traduiront par une courbe qui démarre plus tard.

---
## 6 bis. Correspondance pays entre `region_country` et `dim_country_year`
---

`region_country` est la seule table du modèle sans colonne `Key` : elle n'a pas de dimension année et se joint à `dim_country_year` uniquement sur `Country`. Cette relation n'a jamais été soumise au contrôle des clés orphelines réalisé à la section 6, qui ne portait que sur les jointures par `Key`. Cette section comble cette lacune.

### Vérification des correspondances

La vérification consiste à comparer l'ensemble des pays de `dim_country_year` à ceux de `region_country`. Tout pays présent dans le premier ensemble et absent du second se traduira, une fois les tables importées dans Power BI, par une région vide dans les visuels filtrés par région.

In [16]:
pays_region = set(df_region_country["Country"])
pays_dim = set(df_dim_country_year["Country"])

orphelins = pays_dim - pays_region
print(f"Pays sans région assignée : {len(orphelins)}")
print(orphelins)


Pays sans région assignée : 44
{'French Guyana', 'Faroe Islands', 'Northern Mariana Islands', 'Liechtenstein', 'Western Sahara', 'New Caledonia', 'Bonaire, Sint Eustatius and Saba', 'Guadeloupe', 'North Macedonia', 'Guam', 'China, mainland', 'Montserrat', 'Sint Maarten  (Dutch part)', 'Saint-Martin (French part)', 'Channel Islands', 'Martinique', 'Saint Pierre and Miquelon', 'China, Macao SAR', 'United States Virgin Islands', 'Saint Barthélemy', 'Wallis and Futuna Islands', 'Netherlands Antilles (former)', 'French Polynesia', 'Mayotte', 'Holy See', 'Tokelau', 'Isle of Man', 'China, Hong Kong SAR', 'Turks and Caicos Islands', 'Falkland Islands (Malvinas)', 'Cayman Islands', 'Gibraltar', 'Palestine', 'Anguilla', 'Curaçao', 'Bermuda', 'British Virgin Islands', 'Réunion', 'Greenland', 'Aruba', 'Saint Helena, Ascension and Tristan da Cunha', 'American Samoa', 'China, Taiwan Province of', 'Puerto Rico'}


### Conclusion :

44 pays de `dim_country_year` n'ont pas de correspondance dans `region_country`. Un examen individuel de la liste fait apparaître trois catégories distinctes, analysées et traitées ci-dessous.

In [17]:
pays_dim = set(df_dim_country_year["Country"])
pays_region = set(df_region_country["Country"])

non_matches = pays_dim - pays_region

print(f"Pays dans dim_country_year : {len(pays_dim)}")
print(f"Pays dans region_country : {len(pays_region)}")
print(f"Pays sans région assignée : {len(non_matches)}")
print(sorted(non_matches))

Pays dans dim_country_year : 237
Pays dans region_country : 194
Pays sans région assignée : 44
['American Samoa', 'Anguilla', 'Aruba', 'Bermuda', 'Bonaire, Sint Eustatius and Saba', 'British Virgin Islands', 'Cayman Islands', 'Channel Islands', 'China, Hong Kong SAR', 'China, Macao SAR', 'China, Taiwan Province of', 'China, mainland', 'Curaçao', 'Falkland Islands (Malvinas)', 'Faroe Islands', 'French Guyana', 'French Polynesia', 'Gibraltar', 'Greenland', 'Guadeloupe', 'Guam', 'Holy See', 'Isle of Man', 'Liechtenstein', 'Martinique', 'Mayotte', 'Montserrat', 'Netherlands Antilles (former)', 'New Caledonia', 'North Macedonia', 'Northern Mariana Islands', 'Palestine', 'Puerto Rico', 'Réunion', 'Saint Barthélemy', 'Saint Helena, Ascension and Tristan da Cunha', 'Saint Pierre and Miquelon', 'Saint-Martin (French part)', 'Sint Maarten  (Dutch part)', 'Tokelau', 'Turks and Caicos Islands', 'United States Virgin Islands', 'Wallis and Futuna Islands', 'Western Sahara']


In [18]:
erreur_nom = {"North Macedonia"}

subdivisions_chine = {
    "China, mainland",
    "China, Hong Kong SAR",
    "China, Macao SAR",
    "China, Taiwan Province of",
}

territoires_hors_perimetre = non_matches - erreur_nom - subdivisions_chine

print(f"Erreur de nom : {len(erreur_nom)}")
print(f"Subdivisions chinoises : {len(subdivisions_chine)}")
print(f"Territoires hors périmètre OMS : {len(territoires_hors_perimetre)}")
print(sorted(territoires_hors_perimetre))

Erreur de nom : 1
Subdivisions chinoises : 4
Territoires hors périmètre OMS : 39
['American Samoa', 'Anguilla', 'Aruba', 'Bermuda', 'Bonaire, Sint Eustatius and Saba', 'British Virgin Islands', 'Cayman Islands', 'Channel Islands', 'Curaçao', 'Falkland Islands (Malvinas)', 'Faroe Islands', 'French Guyana', 'French Polynesia', 'Gibraltar', 'Greenland', 'Guadeloupe', 'Guam', 'Holy See', 'Isle of Man', 'Liechtenstein', 'Martinique', 'Mayotte', 'Montserrat', 'Netherlands Antilles (former)', 'New Caledonia', 'Northern Mariana Islands', 'Palestine', 'Puerto Rico', 'Réunion', 'Saint Barthélemy', 'Saint Helena, Ascension and Tristan da Cunha', 'Saint Pierre and Miquelon', 'Saint-Martin (French part)', 'Sint Maarten  (Dutch part)', 'Tokelau', 'Turks and Caicos Islands', 'United States Virgin Islands', 'Wallis and Futuna Islands', 'Western Sahara']


### Conclusion : trois catégories distinctes

- **Erreur de nom (1 pays)** : `North Macedonia` est orthographié `Republic of North Macedonia` dans `region_country`. Le pays existe bel et bien dans le référentiel régional, seul le nom diffère.
- **Nomenclature chinoise (4 lignes)** : `population` détaille la Chine en quatre entités distinctes (continentale, Hong Kong, Macao, Taïwan), chacune couverte par `dim_country_year`, alors que `region_country` ne comporte qu'une ligne `China`. Avant de décider d'un rattachement régional, il faut établir ce que recouvrent exactement ces cinq entités : s'agit-il de quatre territoires à ajouter au référentiel, ou d'un agrégat déjà référencé et de ses composantes ? La section suivante tranche cette question.
- **Hors périmètre OMS (39 territoires)** : le reste correspond à des territoires et dépendances qui ne sont pas des États membres de l'Organisation mondiale de la santé (Groenland, Porto Rico, Palestine, Gibraltar, entre autres). `region_country` ne référence que les 194 États membres, ce n'est pas une lacune mais le périmètre même du référentiel. Reste à savoir ce que ces territoires apportent au modèle : c'est l'objet du contrôle mené plus bas, avant toute décision les concernant.

### Vérification de la nomenclature chinoise

Le premier contrôle consiste à relever, table par table, quelles entités chinoises sont effectivement présentes. Il s'agit de vérifier littéralement cette présence plutôt que de la déduire de l'absence de clé orpheline signalée à la section 6.

In [19]:
for name, df in [
    ("Population", df_population),
    ("Water Services", df_water_services),
    ("Mortality", df_mortality),
    ("Political Stability", df_political_stability),
]:
    pays_table = set(df["Country"])
    presentes = subdivisions_chine & pays_table
    plain_china = "China" in pays_table
    print(f"{name} : {len(presentes)}/4 subdivisions présentes, 'China' seul présent : {plain_china}")
    if presentes != subdivisions_chine:
        print("  Manquantes :", subdivisions_chine - presentes)

Population : 4/4 subdivisions présentes, 'China' seul présent : True
Water Services : 0/4 subdivisions présentes, 'China' seul présent : True
  Manquantes : {'China, Macao SAR', 'China, Hong Kong SAR', 'China, Taiwan Province of', 'China, mainland'}
Mortality : 0/4 subdivisions présentes, 'China' seul présent : True
  Manquantes : {'China, Macao SAR', 'China, Hong Kong SAR', 'China, Taiwan Province of', 'China, mainland'}
Political Stability : 4/4 subdivisions présentes, 'China' seul présent : False


### Conclusion : deux nomenclatures coexistent

Le relevé fait apparaître deux découpages distincts dans les sources :

- `water_services` et `mortality` suivent le découpage de l'OMS et ne connaissent que `China` ;
- `political_stability` suit le même découpage détaillé et ne connaît que les quatre subdivisions, sans ligne `China` ;
- `population` contient les cinq entités à la fois, ce qui explique leur présence dans `dim_country_year`.

C'est ce dernier point qu'il faut éclaircir avant toute décision : `China` est-il un cinquième territoire, distinct des quatre autres, ou leur total ? La comparaison des effectifs de population année par année répond directement à la question.

In [20]:
subdivisions = sorted(subdivisions_chine)

chine = df_population[
    df_population["Country"].isin(["China"] + subdivisions)
    & (df_population["Granularity"] == "Total")
]

comparaison = chine.pivot_table(index="Year", columns="Country", values="Population")
comparaison["Somme des 4 subdivisions"] = comparaison[subdivisions].sum(axis=1)
comparaison["Écart absolu"] = (
    comparaison["China"] - comparaison["Somme des 4 subdivisions"]
).abs()

print("Écart absolu maximal sur la période :", comparaison["Écart absolu"].max())
comparaison[["China", "Somme des 4 subdivisions", "Écart absolu"]].round(1)


Écart absolu maximal sur la période : 2.3283064365386963e-10


Country,China,Somme des 4 subdivisions,Écart absolu
Year,,,
2000,1319551.4,1319551.4,0.0
2001,1328341.2,1328341.2,0.0
2002,1336765.2,1336765.2,0.0
2003,1344908.2,1344908.2,0.0
2004,1352871.0,1352871.0,0.0
2005,1360734.5,1360734.5,0.0
2006,1368528.4,1368528.4,0.0
2007,1376265.6,1376265.6,0.0
2008,1383985.6,1383985.6,0.0


### Conclusion : `China` est l'agrégat des quatre subdivisions

L'écart est nul sur l'ensemble de la période 2000-2018, à l'erreur d'arrondi flottant près, de l'ordre de 10⁻¹⁰ millier d'habitants. `China` n'est pas un cinquième territoire comparable aux quatre autres, c'est leur somme exacte. Les quatre subdivisions sont bien distinctes entre elles, mais elles ne le sont pas de l'agrégat.

La conséquence est directe pour le modèle : rattacher les cinq entités à une même région ferait compter la population chinoise deux fois dans toute agrégation régionale, soit environ 1,4 milliard d'habitants en trop sur la région `Western Pacific`.


### Vérification préalable : que contiennent ces territoires ?

Avant de décider de leur sort, il faut savoir ce qu'ils apportent au modèle. Le contrôle porte sur les 39 territoires hors périmètre identifiés ci-dessus, en laissant de côté la Macédoine du Nord, qui relève d'une erreur de nom, et les quatre subdivisions chinoises, déjà traitées comme composantes d'un agrégat.


In [21]:
for name, df in [
    ("Water Services", df_water_services),
    ("Mortality", df_mortality),
    ("Political Stability", df_political_stability),
    ("Population", df_population),
]:
    n = df["Country"].isin(territoires_hors_perimetre).sum()
    pays = df.loc[df["Country"].isin(territoires_hors_perimetre), "Country"].nunique()
    print(f"  {name:<22} {n:>6} ligne(s), {pays:>3} territoire(s) concerné(s)")


  Water Services              0 ligne(s),   0 territoire(s) concerné(s)
  Mortality                   0 ligne(s),   0 territoire(s) concerné(s)
  Political Stability        79 ligne(s),   5 territoire(s) concerné(s)
  Population               2612 ligne(s),  39 territoire(s) concerné(s)


### Les 43 entités hors référentiel, nommées

La décision d'exclusion doit rester auditable : voici la liste complète des entités présentes dans les sources et absentes du référentiel des 194 États membres de l'OMS. Aucune n'a de ligne dans `water_services` ni dans `mortality`, comme le contrôle ci-dessus vient de l'établir.

**Les 4 subdivisions chinoises**, composantes de l'agrégat `China` déjà traité ci-dessus : `China, Hong Kong SAR`, `China, Macao SAR`, `China, Taiwan Province of`, `China, mainland`.

**Les 39 territoires, dépendances et collectivités** : `American Samoa`, `Anguilla`, `Aruba`, `Bermuda`, `Bonaire, Sint Eustatius and Saba`, `British Virgin Islands`, `Cayman Islands`, `Channel Islands`, `Curaçao`, `Falkland Islands (Malvinas)`, `Faroe Islands`, `French Guyana`, `French Polynesia`, `Gibraltar`, `Greenland`, `Guadeloupe`, `Guam`, `Holy See`, `Isle of Man`, `Liechtenstein`, `Martinique`, `Mayotte`, `Montserrat`, `Netherlands Antilles (former)`, `New Caledonia`, `Northern Mariana Islands`, `Palestine`, `Puerto Rico`, `Réunion`, `Saint Barthélemy`, `Saint Helena, Ascension and Tristan da Cunha`, `Saint Pierre and Miquelon`, `Saint-Martin (French part)`, `Sint Maarten  (Dutch part)`, `Tokelau`, `Turks and Caicos Islands`, `United States Virgin Islands`, `Wallis and Futuna Islands`, `Western Sahara`.

Trois d'entre elles ressemblent à des États souverains et méritent une mention, car la question peut être posée :

- le **Saint-Siège** et la **Palestine** ont un statut d'observateur auprès de l'OMS, ils n'en sont pas États membres ;
- le **Liechtenstein** est membre de l'Organisation des Nations unies, mais n'a jamais adhéré à l'Organisation mondiale de la santé.

Le compte se vérifie : les 194 États membres de l'OMS correspondent aux 193 membres de l'ONU, moins le Liechtenstein, plus les Îles Cook et Nioué. Le référentiel `region_country` contient bien `Cook Islands` et `Niue`, et ne contient pas `Liechtenstein`.

### Où vit la règle de périmètre

Le filtre appliqué ci-dessous ne repose pas sur une liste écrite en dur, mais sur `etats_membres`, lui-même déduit de `region_country`. La conséquence est importante pour la durée de vie du tableau de bord : si l'OMS admet un nouvel État membre, il apparaîtra dans le référentiel publié, et il entrera automatiquement dans le modèle à la préparation suivante, sans qu'une seule ligne de code soit à modifier. La même mécanique vaut pour un territoire qui changerait de statut.


### Traitement retenu

**Macédoine du Nord** : le nom est corrigé dans `region_country`, selon le même principe que les corrections déjà appliquées à la section 6.

**Chine** : `region_country` conserve la seule entité `China`. C'est le choix cohérent avec la nomenclature suivie par les tables de l'OMS, et il évite le double comptage établi ci-dessus.

Ce choix laisse un point à traiter. `political_stability` ne connaissant que les subdivisions, l'entité `China` s'y trouverait sans indice de stabilité politique, alors que la donnée existe : la Chine serait absente du troisième domaine d'expertise, le consulting. `China, mainland` est donc renommé `China` dans cette table, ce qui l'aligne sur la nomenclature retenue pour le modèle, et la colonne `Key` est reconstruite après le renommage, comme en section 6.

L'approximation qui en découle est assumée et documentée : l'indice attribué à `China` est celui de la Chine continentale seule, qui représente 97,8 % de la population de l'agrégat, sans Hong Kong, Macao ni Taïwan. L'indice WGI étant un score de gouvernance national et non une grandeur additive, il ne peut pas être recomposé par moyenne pondérée à partir des quatre subdivisions.

**Territoires hors périmètre OMS : exclusion du modèle.** Le contrôle ci-dessus est sans appel : ces territoires n'ont aucune ligne dans `water_services` ni dans `mortality`. Ils ne portent donc aucune information sur l'accès à l'eau ni sur la mortalité qui lui est liée, c'est-à-dire sur l'objet même du tableau de bord.

Or ce tableau de bord sert à décider où investir, et cette décision repose sur une question précise : la population a-t-elle accès à un service d'eau de base, à un service géré en toute sécurité, ou à aucun des deux ? Une entité qui ne permet pas de répondre à cette question n'a pas sa place dans l'outil : elle allonge les listes de sélection sans jamais alimenter un seul indicateur.

Les quatre tables de données et la table pivot sont donc restreintes aux 194 États membres de l'OMS. Quarante-trois entités sortent du modèle : les trente-neuf territoires contrôlés ci-dessus, et les quatre subdivisions chinoises, dont l'agrégat `China` est conservé. Ce périmètre est également celui de `water_services` et de `region_country` : le modèle devient homogène, et le seul périmètre revendiqué est explicite.

In [22]:
# Erreur de nom : Macédoine du Nord
df_region_country["Country"] = df_region_country["Country"].replace(
    "Republic of North Macedonia", "North Macedonia"
)

# Nomenclature chinoise : political_stability est aligné sur le découpage OMS du modèle
df_political_stability["Country"] = df_political_stability["Country"].replace(
    "China, mainland", "China"
)

# Country a changé sur political_stability : Key doit être reconstruite,
# elle ne se met pas à jour automatiquement.
df_political_stability["Key"] = (
    df_political_stability["Country"] + "_" + df_political_stability["Year"].astype(str)
)

# Périmètre du modèle : les 194 États membres de l'OMS, seuls à porter
# une information d'accès à l'eau. Les autres entités sont exclues.
etats_membres = set(df_region_country["Country"])

avant = {
    "water_services": len(df_water_services),
    "population": len(df_population),
    "political_stability": len(df_political_stability),
    "mortality": len(df_mortality),
    "dim_country_year": len(df_dim_country_year),
}

df_water_services = df_water_services[df_water_services["Country"].isin(etats_membres)]
df_population = df_population[df_population["Country"].isin(etats_membres)]
df_political_stability = df_political_stability[
    df_political_stability["Country"].isin(etats_membres)
]
df_mortality = df_mortality[df_mortality["Country"].isin(etats_membres)]
df_dim_country_year = df_dim_country_year[
    df_dim_country_year["Country"].isin(etats_membres)
]

for nom, n in avant.items():
    apres = len(eval(f"df_{nom}"))
    print(f"{nom:<22} {n:>6} -> {apres:>6} lignes   ({apres - n:+d})")

water_services          10476 ->  10476 lignes   (+0)
population              20914 ->  17922 lignes   (-2992)
political_stability      3526 ->   3393 lignes   (-133)
mortality                 549 ->    549 lignes   (+0)
dim_country_year         4448 ->   3686 lignes   (-762)


In [23]:
assert df_region_country["Country"].is_unique, "Des doublons existent dans region_country"
assert df_political_stability["Key"].is_unique, "La clé n'est pas unique dans political_stability"

pays_dim_final = set(df_dim_country_year["Country"])
assert pays_dim_final <= etats_membres, "Des pays hors périmètre subsistent dans la table pivot"
assert not (etats_membres - set(df_region_country["Country"])), "Référentiel incomplet"

keys_dim = set(df_dim_country_year["Key"])
for name, df in [
    ("Water Services", df_water_services),
    ("Population", df_population),
    ("Political Stability", df_political_stability),
    ("Mortality", df_mortality),
]:
    orphelines = set(df["Key"]) - keys_dim
    assert not orphelines, f"{name} : {len(orphelines)} clé(s) orpheline(s)"

print(f"Périmètre final : {len(pays_dim_final)} pays, "
      f"{df_dim_country_year.shape[0]} couples pays-année")
print("Clés orphelines sur les quatre tables : 0")
print("Pays sans région : 0")
print()
print(df_region_country["Region"].value_counts().to_string())

Périmètre final : 194 pays, 3686 couples pays-année
Clés orphelines sur les quatre tables : 0
Pays sans région : 0

Region
Europe                   53
Africa                   47
Americas                 35
Western Pacific          27
Eastern Mediterranean    21
South-East Asia          11


In [24]:
for name, df in [
    ("Water Services", df_water_services),
    ("Mortality", df_mortality),
    ("Political Stability", df_political_stability),
]:
    cles_chine = df.loc[df["Country"] == "China", "Key"]
    orphelines = set(cles_chine) - set(df_dim_country_year["Key"])
    print(f"{name} : {len(cles_chine)} lignes 'China', {len(orphelines)} clé(s) orpheline(s)")

print("Région assignée à 'China' :",
      df_region_country.loc[df_region_country["Country"] == "China", "Region"].tolist())

Water Services : 54 lignes 'China', 0 clé(s) orpheline(s)
Mortality : 3 lignes 'China', 0 clé(s) orpheline(s)
Political Stability : 18 lignes 'China', 0 clé(s) orpheline(s)
Région assignée à 'China' : ['Western Pacific']


### Conclusion :

Les trois tables qui alimentent les trois domaines d'expertise portent l'entité `China`, sans aucune clé orpheline, et cette entité est rattachée à la région `Western Pacific`.

Le modèle couvre désormais un périmètre unique et explicite : les 194 États membres de l'OMS. Aucune ligne de donnée n'est sans pays, aucun pays n'est sans région, et les six régions du référentiel sont les seules valeurs que l'utilisateur rencontrera dans les segments. Power BI n'a donc aucune modalité vide à créer.

---
## 7. Export, six fichiers CSV séparés  
---

Aucun fichier fusionné n'est produit à cette étape. Chaque table est exportée indépendamment, les jointures seront réalisées dans Power BI, sur la colonne `Key` entre les quatre tables temporelles et `dim_country_year`, et sur la colonne `Country` entre `region_country` et `dim_country_year`.

In [25]:
exports = {
    "water_services_Final.csv": df_water_services,
    "population_Final.csv": df_population,
    "political_stability_Final.csv": df_political_stability,
    "mortality_Final.csv": df_mortality,
    "region_country_Final.csv": df_region_country,
    "dim_country_year.csv": df_dim_country_year,
}

for filename, df in exports.items():
    path = f"../DATA/{filename}"
    df.to_csv(path, index=False, encoding="utf-8-sig")  # utf-8-sig recommandé pour Power BI (accents)
    print(f"{filename}, {df.shape[0]} lignes, {df.shape[1]} colonnes")

water_services_Final.csv, 10476 lignes, 6 colonnes
population_Final.csv, 17922 lignes, 5 colonnes


political_stability_Final.csv, 3393 lignes, 5 colonnes
mortality_Final.csv, 549 lignes, 6 colonnes
region_country_Final.csv, 194 lignes, 2 colonnes
dim_country_year.csv, 3686 lignes, 3 colonnes


---
## 8. Modèle de données et colonnes par domaine, pour Power BI
---

>### Plan de Relations dans Power BI :

| Table | Se relie à | Sur | Cardinalité |
|---|---|---|---|
| `water_services` | `dim_country_year` | `Key` | plusieurs vers un |
| `population` | `dim_country_year` | `Key` | plusieurs vers un |
| `political_stability` | `dim_country_year` | `Key` | plusieurs vers un, naturellement un vers un dans ce cas précis |
| `mortality` | `dim_country_year` | `Key` | plusieurs vers un |
| `region_country` | `dim_country_year` | `Country` | un vers plusieurs |

>### Colonnes à mobiliser par domaine d'expertise :

| Domaine | Colonnes à utiliser | Type de graphique |
|---|---|---|
| Domaine 1, création de services | `pct_basic_water` (Granularity égale Total) et `pct_urban`, calculé depuis `population` | Nuage de points |
| Domaine 2, modernisation | `pct_basic_water` et `pct_safely_managed_water` | Histogramme groupé : les deux taux côte à côte rendent l'écart directement lisible, ce qu'un nuage de points ne permet pas de quantifier aussi simplement |
| Domaine 3, consulting | `pct_basic_water`, `mortality_rate_per_100k` et `Political_Stability` | Nuage de points |
| Vue mondiale et continentale | `pct_basic_water`, `Year` et `Region` | Courbe, diagramme en barres |
| Vue cartographique | `Country` et l'indicateur choisi | Carte choroplèthe |
| Données temporelles | `Year` et tout indicateur | Courbe |
| Mortalité, limitée à 2016 | `mortality_rate_per_100k` et `wash_deaths` | Carte KPI |

Le fichier Power BI contient en outre deux tables techniques, absentes des exports : `Mesure`, qui centralise les mesures DAX, et `Type d'accès à l'eau potable`, un paramètre de champs. Ce ne sont pas des sources métier.

La colonne `pct_urban` n'existe dans aucune des six tables exportées : elle sera calculée directement dans Power BI, sous forme de mesure DAX, à partir de la table `population` filtrée sur les granularités Urban et Total.

>### Gestion de la précision décimale :

Aucun arrondi n'est appliqué en Python à ce stade, ni sur les pourcentages d'accès à l'eau, ni sur le taux de mortalité, ni sur l'indice de stabilité politique. La précision native de chaque source est conservée dans les fichiers exportés, afin que les classements et les calculs s'appuient sur la donnée la plus fidèle disponible. L'affichage à deux décimales, retenu pour la lisibilité du dashboard, sera appliqué dans Power BI au niveau du format des mesures et des colonnes, une couche qui ne modifie pas la donnée sous-jacente.

---
## 9. Documentation, récapitulatif du nettoyage et dictionnaire de données  
---

Ces deux tableaux constituent la synthèse officielle du travail de préparation. Ils sont destinés à être repris dans le support de présentation et dans la page Documentation du dashboard Power BI.

## Récapitulatif du nettoyage opéré :

| Dataset source | Fichier nettoyé | Fichier exporté | Lignes | Granularités conservées | Traitements appliqués | Ajout technique |
|---|---|---|---|---|---|---|
| `BasicAndSafelyManagedDrinkingWaterServices.csv` | `water_services_clean.csv` | `water_services_Final.csv` | 10 476 | Rural, Total, Urban | Plafonnement à 100 % sur les deux colonnes de pourcentage, correction d'une erreur d'arrondi flottant. Valeurs manquantes conservées sur l'accès sécurisé, donnée terrain non collectée dans certains pays et non imputée. Précision native conservée, aucun arrondi | Colonne `Key` (Country_Year), renommage des colonnes de pourcentage |
| `Population.csv` | `population_clean.csv` | `population_Final.csv` | 17 922 | Total, Male, Female, Rural, Urban | Aucun nettoyage nécessaire, aucune valeur manquante ni doublon détecté lors de l'exploration. Restriction du périmètre aux 194 États membres de l'OMS, seuls à porter une information d'accès à l'eau | Colonne `Key` (Country_Year) |
| `PoliticalStability.csv` | `political_stability_clean.csv` | `political_stability_Final.csv` | 3 393 | Total uniquement | Aucune correction appliquée aux valeurs négatives, l'indice de gouvernance utilisé inclut des valeurs négatives par construction. Précision native conservée, aucun arrondi. Alignement de la nomenclature chinoise sur le découpage OMS retenu pour le modèle, `China, mainland` devient `China`. Restriction du périmètre aux 194 États membres de l'OMS, seuls à porter une information d'accès à l'eau | Colonne `Key` (Country_Year), reconstruite après le renommage |
| `MortalityRateAttributedToWater.csv` | `mortality_clean.csv` | `mortality_Final.csv` | 549 | Female, Male, Total | Aucun plafonnement appliqué, le taux pour 100 000 habitants pouvant légitimement dépasser 100. Aucune conversion de type ni arrondi sur `wash_deaths`, une estimation statistique de l'OMS (fraction attribuable) et non un comptage entier, dont la précision native est conservée. Valeurs manquantes conservées et non imputées. Limite reconnue : année 2016 uniquement | Colonne `Key` (Country_Year) |
| `RegionCountry.csv` | `region_country_clean.csv` | `region_country_Final.csv` | 194 | Sans granularité, table fixe | Renommage des colonnes, `COUNTRY (DISPLAY)` devient `Country` et `REGION (DISPLAY)` devient `Region`. Correction d'une erreur de nom, `Republic of North Macedonia` devient `North Macedonia`. Ce référentiel des 194 États membres définit le périmètre du modèle : les 43 entités qu'il ne couvre pas sont exclues, faute de toute donnée d'accès à l'eau | Aucun, jointure sur `Country` seul |
| Dérivée des quatre tables de données | Sans objet | `dim_country_year.csv` | 3 686 | Sans granularité, table technique | Union des combinaisons uniques de `Country` et `Year` des quatre tables de données, afin que la dimension couvre l'intégralité des lignes qu'elle relie, puis restriction aux 194 États membres de l'OMS : 194 pays, 2000 à 2018 | Table créée pour résoudre la non-unicité de `Key` dans les quatre tables de données ; elle devient le pivot central du modèle en étoile |

## Dictionnaire de données :

| Table | Colonne | Type | Unité | Description métier |
|---|---|---|---|---|
| `water_services` | `Key` | texte | sans objet | Clé technique `Country_Year`, relie la table à `dim_country_year` |
| `water_services` | `Country` | texte | sans objet | Nom du pays |
| `water_services` | `Year` | entier | année | de 2000 à 2017 |
| `water_services` | `Granularity` | texte | sans objet | Rural, Total ou Urban |
| `water_services` | `pct_basic_water` | décimal, précision native | pourcentage | Part de la population ayant accès à un service d'eau potable de base, critère minimum retenu par l'OMS |
| `water_services` | `pct_safely_managed_water` | décimal, précision native | pourcentage | Part de la population ayant accès à un service d'eau potable géré en toute sécurité, critère OMS supérieur |
| `population` | `Key` | texte | sans objet | Clé technique `Country_Year` |
| `population` | `Country` | texte | sans objet | Nom du pays |
| `population` | `Year` | entier | année | de 2000 à 2018 |
| `population` | `Granularity` | texte | sans objet | Total, Male, Female, Rural ou Urban |
| `population` | `Population` | décimal | milliers d'habitants | Effectif de population, une valeur de 20 779,953 correspondant à 20,78 millions d'habitants |  
| `population` | `pct_urban` | décimal,précision native | pourcentage | Calculé dans Power BI (Urban / Total). Aucun plafonnement : les composantes Urban et Total de la source ONU sont estimées indépendamment, d'où un léger écart résiduel sur quelques micro-territoires denses. |    
| `political_stability` | `Key` | texte | sans objet | Clé technique `Country_Year` |
| `political_stability` | `Country` | texte | sans objet | Nom du pays. La Chine y figure sous l'entité `China`, alignée sur la nomenclature OMS ; l'indice retenu est celui de la Chine continentale, soit 97,8 % de la population de l'agrégat |
| `political_stability` | `Year` | entier | année | de 2000 à 2018 |
| `political_stability` | `Granularity` | texte | sans objet | Total uniquement |
| `political_stability` | `Political_Stability` | décimal, précision native | indice standardisé, de –3,31 à +1,76 sur le périmètre retenu | Indice de stabilité politique de la Banque mondiale (WGI). Score standardisé recentré chaque année sur la moyenne mondiale, comprise entre environ –2,5 et +2,5 (WGI 2025 Methodology Revision, p. 18) : une valeur plus élevée traduit une meilleure stabilité, et le négatif signale simplement une position sous la moyenne mondiale |
| `mortality` | `Key` | texte | sans objet | Clé technique `Country_Year` |
| `mortality` | `Country` | texte | sans objet | Nom du pays |
| `mortality` | `Year` | entier | année | 2016 uniquement, limite méthodologique de la source OMS |
| `mortality` | `Granularity` | texte | sans objet | Female, Male ou Total |
| `mortality` | `mortality_rate_per_100k` | décimal, précision native | décès pour 100 000 habitants | Taux de mortalité attribuable à un service d'eau non sécurisé, indicateur OMS SDG 3.9.2 |
| `mortality` | `wash_deaths` | décimal, précision native | nombre de décès (estimation) | Nombre de décès attribuables à l'eau non sécurisée, fraction attribuable estimée par l'OMS (Comparative Risk Assessment), pas un comptage direct |
| `region_country` | `Country` | texte | sans objet | Nom du pays, clé de jointure vers `dim_country_year` |
| `region_country` | `Region` | texte | sans objet | Une des six régions officielles de découpage de l'OMS, à distinguer des continents. Ce référentiel définit le périmètre du modèle |
| `dim_country_year` | `Key` | texte | sans objet | Clé primaire unique, `Country_Year`, pivot du modèle en étoile. Couvre l'union des couples pays-année des quatre tables de données, restreinte aux 194 États membres de l'OMS |
| `dim_country_year` | `Country` | texte | sans objet | Nom du pays |
| `dim_country_year` | `Year` | entier | année | de 2000 à 2018 |